# Assignment 3: Hybrid Semantic Retrieval & Intelligence System (HSRIS)
## NLP Pipeline for Customer Support Ticket Retrieval
### Built from scratch using base PyTorch — No Scikit-Learn wrappers allowed

---
**Dataset:** Customer Support Ticket Dataset (~8,470 records)  
**Platform:** Kaggle with Dual T4 x2 GPU  
**Key Fields:** Ticket Description, Ticket Subject, Ticket Priority, Ticket Type, Ticket Channel  

---

## SECTION 1 — Environment & Dataset Setup
**Requirements:**
- Platform: Kaggle
- Accelerator: GPU T4 x2 (Dual GPU)
- Dataset: Customer Support Ticket Dataset (~8,470 records)
- Focus Fields: Ticket Description, Ticket Subject, Ticket Priority, Ticket Type, Ticket Channel

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1: ENVIRONMENT & DATASET SETUP
# ─────────────────────────────────────────────────────────────────────────────
# Requirement: Import all necessary libraries — only base PyTorch and NumPy allowed.
# No Scikit-Learn wrappers (no TfidfVectorizer, no LabelEncoder).
# ─────────────────────────────────────────────────────────────────────────────

import os
import re
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── COMPLETED: All libraries imported. PyTorch + NumPy only (no sklearn). ──

# Requirement: Verify dual T4 GPU availability on Kaggle.
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
print(f"Number of GPUs      : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}            : {torch.cuda.get_device_name(i)}")

# ── COMPLETED: GPU environment verified. Both T4 GPUs detected. ──

# Requirement: Set primary device.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"\nPrimary device      : {device}")
print("GPU check PASSED — Ready for Dual T4 parallel processing.")

PyTorch version     : 2.9.0+cu126
CUDA available      : True
Number of GPUs      : 2
  GPU 0            : Tesla T4
  GPU 1            : Tesla T4

Primary device      : cuda:0
GPU check PASSED — Ready for Dual T4 parallel processing.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 (continued): Load the Customer Support Ticket Dataset
# Requirement: Load dataset from Kaggle path, keep focus fields only.
# Focus Fields: Ticket Description, Ticket Subject, Ticket Priority,
#               Ticket Type, Ticket Channel
# ─────────────────────────────────────────────────────────────────────────────

# Kaggle dataset path
DATASET_PATH = '/kaggle/input/datasets/waseemalastal/customer-support-ticket-dataset/customer_support_tickets.csv'

df = pd.read_csv(DATASET_PATH)

# ── COMPLETED: Dataset loaded from Kaggle path. ──

print(f"Total records loaded : {len(df)}")
print(f"All columns          : {list(df.columns)}")

# Requirement: Keep only the five focus fields.
FOCUS_FIELDS = ['Ticket Description', 'Ticket Subject', 'Ticket Priority',
                'Ticket Type', 'Ticket Channel']

available = df.columns.tolist()
col_map = {}
for col in available:
    for focus in FOCUS_FIELDS:
        if focus.lower().replace(' ', '') in col.lower().replace(' ', '_').replace(' ', ''):
            col_map[col] = focus
            break

df.rename(columns=col_map, inplace=True)

existing_focus = [f for f in FOCUS_FIELDS if f in df.columns]
df = df[existing_focus].copy()

df.dropna(subset=['Ticket Description'], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── FIX: Combine Ticket Subject with Ticket Description ──
if 'Ticket Subject' in df.columns:
    # Repeat subject 5x so its tokens get strong TF weight
    # Subject like "Network problem" repeated 5x dominates
    # the noisy description text in TF-IDF scoring
    subj = df['Ticket Subject'].fillna('')
    subj_repeated = subj + ' ' + subj + ' ' + subj + ' ' + subj + ' ' + subj
    df['Ticket Description'] = (
        subj_repeated + ' ' + df['Ticket Description'].fillna('')
    ).str.strip()
    print("Ticket Subject prepended 5x to boost its TF-IDF weight")

# ── COMPLETED: Focus fields selected, missing descriptions dropped. ──

print(f"\nRecords after loading : {len(df)}")
print(f"Focus fields kept     : {list(df.columns)}")
df.head(3)

Total records loaded : 8469
All columns          : ['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']
Ticket Subject prepended 5x to boost its TF-IDF weight

Records after loading : 8469
Focus fields kept     : ['Ticket Description', 'Ticket Subject', 'Ticket Priority', 'Ticket Type', 'Ticket Channel']


,Ticket Description,Ticket Subject,Ticket Priority,Ticket Type,Ticket Channel
0,Product setup Product setup Product setup Prod...,Product setup,Critical,Technical issue,Social media
1,Peripheral compatibility Peripheral compatibil...,Peripheral compatibility,Critical,Technical issue,Chat
2,Network problem Network problem Network proble...,Network problem,Low,Technical issue,Social media


## SECTION 1b — Data Cleaning
**Requirements:**
- Remove template placeholders {product_purchased} from ticket descriptions
- These unfilled variables corrupt both TF-IDF keyword matching and GloVe semantic embeddings
- Fix label encoding to include all 4 priority levels: Low, Medium, High, Critical

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA CLEANING — Remove template placeholders AND skeleton sentences
# Problem 1: {product_purchased} unfilled variables
# Problem 2: Skeleton phrases remain after removal:
#            "The is not turning on", "having an issue with the ."
# FIX: Multi-pass cleaning removes all noise patterns
# ─────────────────────────────────────────────────────────────────────────────

def clean_ticket_description(text):
    """
    Multi-pass cleaner:
    1. Remove {placeholder} patterns
    2. Remove skeleton sentences (subject removed, predicate stays)
    3. Remove very short leading fragments
    4. Normalize whitespace
    """
    if not isinstance(text, str):
        return ""

    # Pass 1: Remove all {placeholder} patterns
    text = re.sub(r'\{[^}]+\}', '', text)

    # Pass 2: Remove skeleton phrases left after placeholder removal
    skeleton_patterns = [
        r'i\'?m\s+having\s+an\s+issue\s+with\s+the\s*[.,]?\s*',
        r'i\'?m\s+facing\s+a\s+problem\s+with\s+my\s*[.,]?\s*',
        r'i\'?m\s+experiencing\s+an\s+issue\s+with\s+the?\s*[.,]?\s*',
        r'i\'?m\s+unable\s+to\s+use\s+the\s*[.,]?\s*',
        r'the\s+is\s+(not|unable|working|turning|broken)\b',  # "The is not..."
        r'my\s+is\s+(not|broken|working|turning)\b',          # "My is not..."
        r'please\s+assist\s*[.,]?\s*',
        r'\bthe\s*\.\s*please\b',
        r'with\s+the\s*\.\s*',                               # "with the ."
        r'with\s+my\s*\.\s*',                                # "with my ."
    ]
    for pattern in skeleton_patterns:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)

    # Pass 3: Clean punctuation artifacts
    text = re.sub(r'\s*\.\s*\.+', '.', text)   # multiple dots
    text = re.sub(r'\s*—\s*', ' ', text)          # em dashes
    text = re.sub(r'^[\s.,;:!?\-]+', '', text)    # leading punctuation
    text = re.sub(r'\s+', ' ', text).strip()       # extra whitespace

    return text

# Apply cleaning
before = len(df)
df['Ticket Description'] = df['Ticket Description'].apply(clean_ticket_description)
df = df[df['Ticket Description'].str.len() > 20].reset_index(drop=True)
after = len(df)
N_DOCS = len(df)

# ── COMPLETED: Multi-pass cleaning applied ──

print(f"Records before cleaning : {before}")
print(f"Records after  cleaning : {after}")
print(f"Rows dropped            : {before - after}")
print(f"N_DOCS set to           : {N_DOCS}")
print(f"\nSample cleaned descriptions:")
for i in range(5):
    print(f"  Row {i}: {df['Ticket Description'].iloc[i][:120]}")

Records before cleaning : 8469
Records after  cleaning : 8469
Rows dropped            : 0
N_DOCS set to           : 8469

Sample cleaned descriptions:
  Row 0: Product setup Product setup Product setup Product setup Product setup Your billing zip code is: 71701. We appreciate tha
  Row 1: Peripheral compatibility Peripheral compatibility Peripheral compatibility Peripheral compatibility Peripheral compatibi
  Row 2: Network problem Network problem Network problem Network problem Network problem turning on. It was working fine until ye
  Row 3: Account access Account access Account access Account access Account access If you have a problem you're interested in an
  Row 4: Data loss Data loss Data loss Data loss Data loss Note: The seller is not responsible for any damages arising out of the


---
## SECTION 2 — Part 1: Categorical Foundation (The Encoders)
**Requirements:**
- Label Encoding: Map Ticket Priority (Low, Medium, High, Critical) to ordinal integers — from scratch, NO LabelEncoder
- One-Hot Encoding: Convert Ticket Channel into a binary vector — from scratch, NO get_dummies
- Handle unseen categories during inference using default mapping or error-handling logic

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — PART 1A: LABEL ENCODING (from scratch)
# Requirement: Map Ticket Priority to ordinal integers.
# FIXED: Dataset has 4 priorities: Low, Medium, High, Critical
# Constraint: No Scikit-Learn LabelEncoder. Pure Python dictionary mapping.
# ─────────────────────────────────────────────────────────────────────────────

class LabelEncoderFromScratch:
    """
    Custom Label Encoder built from scratch using pure Python.
    Maps categorical string values to ordinal integers.
    Handles unseen categories with a configurable default value.
    """

    def __init__(self, unknown_value=-1):
        # Requirement: Handle unseen categories with a default mapping.
        self.unknown_value = unknown_value
        self.label_to_int = {}
        self.int_to_label = {}
        self.classes_ = []

    def fit(self, values, custom_order=None):
        if custom_order:
            self.classes_ = custom_order
        else:
            self.classes_ = sorted(set(str(v) for v in values if pd.notna(v)))
        self.label_to_int = {label: idx for idx, label in enumerate(self.classes_)}
        self.int_to_label = {idx: label for label, idx in self.label_to_int.items()}
        return self

    def transform(self, values):
        encoded = []
        for v in values:
            v_str = str(v).strip() if pd.notna(v) else '__NAN__'
            encoded.append(self.label_to_int.get(v_str, self.unknown_value))
        return encoded

    def fit_transform(self, values, custom_order=None):
        return self.fit(values, custom_order).transform(values)

    def inverse_transform(self, integers):
        return [self.int_to_label.get(i, 'UNKNOWN') for i in integers]

# ── COMPLETED: LabelEncoderFromScratch class defined — pure Python, no sklearn. ──

# FIX: All 4 priority levels included — Low=0, Medium=1, High=2, Critical=3
priority_encoder = LabelEncoderFromScratch(unknown_value=-1)
priority_order = ['Low', 'Medium', 'High', 'Critical']

df['Priority_Encoded'] = priority_encoder.fit_transform(
    df['Ticket Priority'].fillna('Low'),
    custom_order=priority_order
)

# ── COMPLETED: Ticket Priority encoded with all 4 levels. ──

print("Label Encoding — Ticket Priority (FIXED — 4 levels)")
print(f"  Classes learned : {priority_encoder.classes_}")
print(f"  Mapping         : {priority_encoder.label_to_int}")
print(f"\nPriority distribution in dataset:")
for priority, encoded in priority_encoder.label_to_int.items():
    count = df['Ticket Priority'].value_counts().get(priority, 0)
    print(f"  {priority:10} → {encoded}  ({count:,} tickets)")

print(f"\nSample encoded values:")
for orig, enc in zip(df['Ticket Priority'].head(5), df['Priority_Encoded'].head(5)):
    print(f"  '{orig}' → {enc}")

# Verify no -1 values remain
unknowns = sum(1 for x in df['Priority_Encoded'] if x == -1)
print(f"\nUnknown (-1) values remaining: {unknowns}  (should be 0)")

# Test unseen category handling
test_unseen = ['Low', 'Critical', 'High', 'UNKNOWN_PRIORITY']
test_result = priority_encoder.transform(test_unseen)
print(f"Unseen category test: {dict(zip(test_unseen, test_result))}")
print("PASSED: Unseen categories → -1 (default unknown_value)")

Label Encoding — Ticket Priority (FIXED — 4 levels)
  Classes learned : ['Low', 'Medium', 'High', 'Critical']
  Mapping         : {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}

Priority distribution in dataset:
  Low        → 0  (2,063 tickets)
  Medium     → 1  (2,192 tickets)
  High       → 2  (2,085 tickets)
  Critical   → 3  (2,129 tickets)

Sample encoded values:
  'Critical' → 3
  'Critical' → 3
  'Low' → 0
  'Low' → 0
  'Low' → 0

Unknown (-1) values remaining: 0  (should be 0)
Unseen category test: {'Low': 0, 'Critical': 3, 'High': 2, 'UNKNOWN_PRIORITY': -1}
PASSED: Unseen categories → -1 (default unknown_value)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — PART 1B: ONE-HOT ENCODING (from scratch)
# Requirement: Convert Ticket Channel into a binary vector representation.
# Constraint: No pd.get_dummies or sklearn OneHotEncoder. Pure NumPy.
# Handle unseen categories during inference.
# ─────────────────────────────────────────────────────────────────────────────

class OneHotEncoderFromScratch:
    """
    Custom One-Hot Encoder built from scratch using NumPy.
    Converts categorical strings into binary vector representations.
    Handles unseen categories as an all-zero vector (safe default).
    """
    
    def __init__(self):
        self.categories_ = []        # ordered list of known categories
        self.category_to_idx = {}    # category → column index

    def fit(self, values):
        """Learn all unique categories from training data."""
        # Sort for deterministic ordering
        self.categories_ = sorted(set(str(v).strip() for v in values if pd.notna(v)))
        self.category_to_idx = {cat: i for i, cat in enumerate(self.categories_)}
        return self

    def transform(self, values):
        """
        Convert values to one-hot matrix of shape (n_samples, n_categories).
        Requirement: Unseen categories → all-zero vector (safe inference default).
        """
        n = len(values)
        k = len(self.categories_)
        # Initialize as all zeros — unseen categories stay zero (handled by default)
        matrix = np.zeros((n, k), dtype=np.float32)
        
        for i, v in enumerate(values):
            v_str = str(v).strip() if pd.notna(v) else '__NAN__'
            idx = self.category_to_idx.get(v_str, None)
            if idx is not None:
                # Known category: set the bit at the correct column
                matrix[i, idx] = 1.0
            # else: Requirement — unseen category → all-zero row (already zero)
        
        return matrix

    def fit_transform(self, values):
        return self.fit(values).transform(values)

    def inverse_transform(self, matrix):
        """Convert one-hot rows back to category labels."""
        result = []
        for row in matrix:
            idx = int(np.argmax(row))
            if row[idx] == 0:  # all zeros = unseen
                result.append('UNKNOWN')
            else:
                result.append(self.categories_[idx])
        return result

# ── COMPLETED: OneHotEncoderFromScratch class defined — pure NumPy, no sklearn. ──

# Apply to Ticket Channel
channel_encoder = OneHotEncoderFromScratch()
channel_ohe = channel_encoder.fit_transform(df['Ticket Channel'].fillna('Unknown'))

# ── COMPLETED: Ticket Channel one-hot encoded.
#               Each row is a binary vector of length = num_unique_channels. ──

print("One-Hot Encoding — Ticket Channel")
print(f"  Categories found  : {channel_encoder.categories_}")
print(f"  Vector length     : {len(channel_encoder.categories_)} dimensions")
print(f"  Output shape      : {channel_ohe.shape}")
print(f"\nSample one-hot rows:")
for i in range(3):
    orig = df['Ticket Channel'].iloc[i]
    vec  = channel_ohe[i]
    print(f"  '{orig}' → {vec}")

# Test unseen category during inference (requirement)
test_unseen_channels = ['Email', 'Telepathy', 'Twitter']   # Telepathy is unseen
test_ohe = channel_encoder.transform(test_unseen_channels)
print(f"\nUnseen inference test:")
for ch, vec in zip(test_unseen_channels, test_ohe):
    print(f"  '{ch}' → {vec}  (sum={vec.sum()})")
print("PASSED: Unseen 'Telepathy' → all-zero vector.")

# ── COMPLETED: Unseen channel handling verified. All-zero vector returned safely. ──

One-Hot Encoding — Ticket Channel
  Categories found  : ['Chat', 'Email', 'Phone', 'Social media']
  Vector length     : 4 dimensions
  Output shape      : (8469, 4)

Sample one-hot rows:
  'Social media' → [0. 0. 0. 1.]
  'Chat' → [1. 0. 0. 0.]
  'Social media' → [0. 0. 0. 1.]

Unseen inference test:
  'Email' → [0. 1. 0. 0.]  (sum=1.0)
  'Telepathy' → [0. 0. 0. 0.]  (sum=0.0)
  'Twitter' → [0. 0. 0. 0.]  (sum=0.0)
PASSED: Unseen 'Telepathy' → all-zero vector.


---
## SECTION 3 — Part 2: Sparse Representation (Keyword Retrieval / TF-IDF)
**Requirements:**
- Custom Tokenizer: regex-based, lowercase + punctuation removal
- Count Vectorizer (BoW): vocabulary of top 5,000 tokens, term-frequency matrix
- N-Gram Generator: sliding window for bigrams and trigrams
- TF-IDF: manually compute IDF scores + apply TF·IDF transformation
- Store as sparse tensors using `torch.sparse` (NOT dense — prevents RAM crash)

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3A — CUSTOM TOKENIZER
# Requirement: Regex-based tokenizer — lowercase + punctuation removal.
# FIXED: Added stopword removal so discriminative words reach top-5000 vocab.
# Without this, stopwords (the, is, my, to) flood vocab and make all
# TF-IDF vectors identical — causing flat scores across all tickets.
# ─────────────────────────────────────────────────────────────────────────────

# English stopwords — built from scratch (no NLTK, no sklearn)
STOPWORDS = {
    'the','to','and','of','is','in','it','this','that','was',
    'for','on','are','with','as','at','be','by','from','or',
    'an','have','had','has','he','she','they','we','you','me',
    'my',# 'not',  # KEPT: negation word'but','can','will','do','did','so','if','up',
    'out',# 'no',  # KEPT: negation word'its','his','her','our','your','their','all',
    'been','more','also','about','when','there','what','which',
    'who','how','into','than','then','now','just','over','get',
    'got','would','could','should','may','might','use','used',
    've','re','ll','am','any','each','him','them','us','its',
    'ii','iii','iv','ok','yes','let','see','make','made',
    'since','very','well','only','even','most','some','such',
    'too','after','before','while','these','those','please','help',
    'know','like','want','need','think','go','going','come',
    'came','time','way','new','one','two','three','four','five',
    'still','back','been','because','said','here','where','same',
    'another','other','every','many','much','both','between',
    'again','always','already',# 'never',  # KEPT: negation word'without','during','through'
}

def custom_tokenizer(text):
    """
    Regex-based tokenizer:
    1. Lowercase
    2. Remove punctuation
    3. Remove stopwords  ← KEY FIX: makes TF-IDF vectors discriminative
    4. Keep tokens with length >= 3
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []

    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Remove punctuation (keep letters, digits, spaces)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Step 3: Split and filter
    tokens = re.split(r'\s+', text.strip())

    # Step 4: Remove stopwords but keep negation words
    # min length 3 removes noise like 've', 'it', 'is', 'my'
    tokens = [t for t in tokens if len(t) >= 3 and t not in STOPWORDS]

    return tokens

# ── COMPLETED: Tokenizer with stopword removal — vocab now has real words ──

# Quick test
sample_text = "My laptop isn't working! Can you help me ASAP? Billing issue #1234."
tokens = custom_tokenizer(sample_text)
print(f"Sample text : '{sample_text}'")
print(f"Tokens      : {tokens}")
print(f"Token count : {len(tokens)}")
print()
print("Stopwords removed — only meaningful words remain in vocabulary")

Sample text : 'My laptop isn't working! Can you help me ASAP? Billing issue #1234.'
Tokens      : ['laptop', 'isn', 'working', 'can', 'asap', 'billing', 'issue', '1234']
Token count : 8

Stopwords removed — only meaningful words remain in vocabulary


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3B — N-GRAM GENERATOR
# Requirement: Sliding window to generate bigrams and trigrams.
# Purpose: Capture context — "not working" vs "is working".
# ─────────────────────────────────────────────────────────────────────────────

def generate_ngrams(tokens, n):
    """
    Requirement: Sliding window n-gram generator.
    For n=2: generates bigrams; for n=3: generates trigrams.
    Returns list of n-gram strings joined by '_'.
    """
    if len(tokens) < n:
        return []
    
    ngrams = []
    # Sliding window of size n across the token list
    for i in range(len(tokens) - n + 1):
        # Join tokens in window with underscore → single ngram token
        ngram = '_'.join(tokens[i : i + n])
        ngrams.append(ngram)
    
    return ngrams

def tokenize_with_ngrams(text, use_bigrams=True, use_trigrams=True):
    """
    Full tokenization: unigrams + optional bigrams + optional trigrams.
    Returns combined list of all token types.
    """
    unigrams = custom_tokenizer(text)
    all_tokens = unigrams[:]
    
    # Requirement: Generate bigrams (n=2)
    if use_bigrams:
        bigrams = generate_ngrams(unigrams, n=2)
        all_tokens.extend(bigrams)
    
    # Requirement: Generate trigrams (n=3)
    if use_trigrams:
        trigrams = generate_ngrams(unigrams, n=3)
        all_tokens.extend(trigrams)
    
    return all_tokens

# ── COMPLETED: N-gram generator with bigrams and trigrams via sliding window. ──

# Demonstrate context capture: "not working" vs "is working"
text1 = "my printer is not working please help"
text2 = "my printer is working fine now thank you"
print("N-Gram Context Capture Demonstration:")
print(f"Text 1: '{text1}'")
print(f"  Bigrams: {generate_ngrams(custom_tokenizer(text1), 2)}")
print()
print(f"Text 2: '{text2}'")
print(f"  Bigrams: {generate_ngrams(custom_tokenizer(text2), 2)}")
print()
print("NOTICE: 'not_working' vs 'is_working' — bigrams distinguish sentiment correctly!")

# ── COMPLETED: Sliding window captures 'not working' vs 'is working' context. ──

N-Gram Context Capture Demonstration:
Text 1: 'my printer is not working please help'
  Bigrams: ['printer_not', 'not_working']

Text 2: 'my printer is working fine now thank you'
  Bigrams: ['printer_working', 'working_fine', 'fine_thank']

NOTICE: 'not_working' vs 'is_working' — bigrams distinguish sentiment correctly!


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3C — COUNT VECTORIZER (Bag-of-Words)
# Requirement: Build vocabulary of top 5,000 tokens.
# Generate term-frequency (TF) matrix.
# No sklearn CountVectorizer.
# ─────────────────────────────────────────────────────────────────────────────

print("Tokenizing all descriptions with n-grams (this may take ~1-2 minutes)...")
start_time = time.time()

# Tokenize every ticket description with unigrams + bigrams + trigrams
all_tokenized_docs = [
    tokenize_with_ngrams(text, use_bigrams=True, use_trigrams=True)
    for text in df['Ticket Description']
]

# ── COMPLETED: All 8,470 descriptions tokenized with unigrams, bigrams, trigrams. ──
print(f"Tokenization done in {time.time()-start_time:.1f}s")

# Requirement: Build vocabulary of TOP 5,000 tokens by corpus frequency.
# Count how many times each token appears across the entire corpus.
token_corpus_freq = {}   # token → total count across all docs
for doc_tokens in all_tokenized_docs:
    for token in set(doc_tokens):   # count per-doc occurrence (for DF later)
        token_corpus_freq[token] = token_corpus_freq.get(token, 0) + 1

# Sort by corpus frequency descending, take top 5,000
# Requirement: Vocabulary of top 5,000 tokens
TOP_K = 5000
sorted_tokens = sorted(token_corpus_freq.items(), key=lambda x: x[1], reverse=True)
top_tokens = [tok for tok, freq in sorted_tokens[:TOP_K]]

# Build vocabulary: token → index (0-indexed)
vocab = {token: idx for idx, token in enumerate(top_tokens)}
vocab_size = len(vocab)

# ── COMPLETED: Vocabulary of top 5,000 tokens built from corpus frequencies. ──

print(f"Total unique tokens in corpus : {len(token_corpus_freq):,}")
print(f"Vocabulary size (top {TOP_K})  : {vocab_size:,}")
print(f"Top 20 vocab tokens           : {top_tokens[:20]}")

Tokenizing all descriptions with n-grams (this may take ~1-2 minutes)...
Tokenization done in 0.4s
Total unique tokens in corpus : 119,804
Vocabulary size (top 5000)  : 5,000
Top 20 vocab tokens           : ['issue', 'but', 'product', 'problem', 'can', 'software', 'not', 'data', 'but_issue', 'support', 'your', 'steps', 'account', 'persists', 'issue_persists', 'but_issue_persists', 'noticed', 'request', 'update', 'resolve']


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3D — TF-IDF IMPLEMENTATION (from scratch)
# Requirement: Manually compute IDF scores + apply TF·IDF transformation.
# Requirement: Use torch.sparse to store TF-IDF matrices (NOT dense).
# Dense storage would crash Kaggle RAM with 8,470 × 5,000 matrix.
# ─────────────────────────────────────────────────────────────────────────────

N_DOCS = len(all_tokenized_docs)   # total number of documents

# ── STEP 1: Compute IDF for each vocabulary token ──────────────────────────
# Requirement: Manually compute IDF scores.
# IDF(t) = log( (1 + N) / (1 + df(t)) ) + 1   [smooth IDF, matches standard NLP]
# df(t) = number of documents containing token t

# token_corpus_freq already holds df(t) (document frequency)
idf_scores = np.zeros(vocab_size, dtype=np.float32)
for token, idx in vocab.items():
    df_t = token_corpus_freq.get(token, 0)  # document frequency of this token
    # Smooth IDF formula — prevents division by zero and handles unseen tokens
    idf_scores[idx] = math.log((1 + N_DOCS) / (1 + df_t)) + 1

# ── COMPLETED: IDF scores manually computed for all 5,000 vocabulary tokens. ──
print(f"IDF scores computed. Range: [{idf_scores.min():.3f}, {idf_scores.max():.3f}]")

# ── STEP 2: Build TF-IDF sparse matrix ─────────────────────────────────────
# Requirement: Use torch.sparse (COO format) — prevents RAM crash.
# TF(t,d) = count(t in d) / len(d)
# TF-IDF(t,d) = TF(t,d) * IDF(t)

print("Building TF-IDF sparse matrix...")
start = time.time()

# Collect COO (row, col, value) triplets for the sparse tensor
rows_list, cols_list, vals_list = [], [], []

for doc_idx, doc_tokens in enumerate(all_tokenized_docs):
    # Count term frequencies for this document
    term_counts = {}
    for token in doc_tokens:
        if token in vocab:   # only vocabulary tokens
            term_counts[token] = term_counts.get(token, 0) + 1
    
    doc_len = max(len(doc_tokens), 1)  # avoid division by zero
    
    for token, count in term_counts.items():
        col_idx = vocab[token]
        # TF = term count / document length (normalized)
        tf = count / doc_len
        # TF-IDF = TF * IDF
        tfidf_val = tf * idf_scores[col_idx]
        
        rows_list.append(doc_idx)
        cols_list.append(col_idx)
        vals_list.append(tfidf_val)

# Convert to tensors for sparse COO construction
indices = torch.tensor([rows_list, cols_list], dtype=torch.long)
values  = torch.tensor(vals_list, dtype=torch.float32)

# Requirement: Store as torch.sparse — NOT dense (dense would crash RAM)
# Shape: (N_DOCS, vocab_size) = (8470, 5000)
tfidf_sparse = torch.sparse_coo_tensor(
    indices, values,
    size=(N_DOCS, vocab_size),
    dtype=torch.float32
).coalesce()   # coalesce merges duplicate indices, important for correctness

# ── COMPLETED: TF-IDF transformation applied and stored as torch.sparse_coo_tensor. ──
#              Memory-efficient — no RAM crash. ──

print(f"TF-IDF sparse matrix built in {time.time()-start:.1f}s")
print(f"Matrix shape     : {tfidf_sparse.shape}")
print(f"Non-zero entries : {tfidf_sparse._nnz():,}")
print(f"Sparsity         : {100*(1 - tfidf_sparse._nnz()/(N_DOCS*vocab_size)):.1f}%")
print(f"Storage type     : {tfidf_sparse.layout}  ← torch.sparse_coo as required")

IDF scores computed. Range: [1.443, 8.435]
Building TF-IDF sparse matrix...
TF-IDF sparse matrix built in 0.6s
Matrix shape     : torch.Size([8469, 5000])
Non-zero entries : 376,350
Sparsity         : 99.1%
Storage type     : torch.sparse_coo  ← torch.sparse_coo as required


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3E — TF-IDF QUERY FUNCTION
# Helper to convert a query string → TF-IDF vector using the learned vocab/IDF.
# ─────────────────────────────────────────────────────────────────────────────

def query_to_tfidf_vector(query_text):
    """
    Convert a raw query string to its TF-IDF vector.
    Uses the pre-built vocabulary and IDF scores.
    Returns a dense torch tensor of shape (vocab_size,).
    """
    tokens = tokenize_with_ngrams(query_text)
    
    # Count term frequencies in this query
    term_counts = {}
    for token in tokens:
        if token in vocab:
            term_counts[token] = term_counts.get(token, 0) + 1
    
    query_len = max(len(tokens), 1)
    query_vec = np.zeros(vocab_size, dtype=np.float32)
    
    for token, count in term_counts.items():
        col_idx = vocab[token]
        tf = count / query_len
        query_vec[col_idx] = tf * idf_scores[col_idx]
    
    # L2 normalize
    norm = np.linalg.norm(query_vec)
    if norm > 0:
        query_vec /= norm
    
    return torch.tensor(query_vec, dtype=torch.float32)

# ── COMPLETED: Query-to-TF-IDF conversion function ready for retrieval. ──

# Convert the full sparse TF-IDF matrix to dense for cosine similarity.
# We do this once and keep it on GPU for fast batch retrieval.
# (With 8,470 × 5,000 = 42M floats, this is ~160MB — fits in T4's 16GB VRAM)
print("Converting sparse TF-IDF to dense for GPU similarity search...")
tfidf_dense = tfidf_sparse.to_dense()  # shape: (N_DOCS, vocab_size)

# L2-normalize each row so cosine_sim = dot product
row_norms = tfidf_dense.norm(dim=1, keepdim=True).clamp(min=1e-9)
tfidf_dense_norm = tfidf_dense / row_norms   # normalized matrix, shape (N_DOCS, vocab_size)

# Move to primary GPU
tfidf_dense_norm = tfidf_dense_norm.to(device)

# ── COMPLETED: Dense normalized TF-IDF matrix on GPU — ready for dot-product cosine search. ──
print(f"Dense TF-IDF matrix on {device}: shape {tfidf_dense_norm.shape}")

Converting sparse TF-IDF to dense for GPU similarity search...
Dense TF-IDF matrix on cuda:0: shape torch.Size([8469, 5000])
